# Polymarket Global — do skilled wallets actually exist?

Run in **Colab**. Every Polymarket host is blocked from the Claude Code sandbox
(403 on CONNECT), so the crawl runs here.

**This notebook answers one question and defers everything else.** Execution and
the US/Global geo split are real problems, but they only matter if the answer
here is yes. The question:

> Do wallets exist with an edge large enough to be told apart from the luckiest
> trader in a population of thousands?

The bar, from `POLYMARKET.md`: **~15.7 cents/share over 200+ trades** when
scanning 10,000 wallets. To merely be profitable to copy needs ~4.8c. The
identification bar is three times the profitability bar, and that gap is the
whole problem.

**Budget: free.** The APIs are public and keyless.

In [ ]:
!git clone -q https://github.com/parsiqman/flow-signal.git 2>/dev/null || true
%cd flow-signal
!pip install -q pandas numpy matplotlib

import sys; sys.path.insert(0, 'src')
import numpy as np, pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

from polymarket import client, wallets, execution
print('ready')

## 1. The one rule that must not be broken

Polymarket publishes a **public leaderboard**. Seeding candidate wallets from it
selects on the outcome variable — you would rank traders by past profit inside a
set already filtered for past profit, then "discover" that past profit predicts
past profit. The numbers would look superb and mean nothing, and **nothing in the
pipeline would appear to fail.**

So wallets are enumerated by *market participation*: sample resolved markets,
take everyone who traded, keep them all regardless of how they did.

`discover_by_leaderboard()` exists and raises, so the trap is recorded in code.

In [ ]:
try:
    client.discover_by_leaderboard()
except NotImplementedError as e:
    print('correctly refused:', e)

## 2. Fetch resolved markets

Caching to disk makes the crawl resumable and reproducible — a run interrupted
halfway must not resume against a different slice of data.

In [ ]:
cfg = client.ClientConfig(cache_dir='/content/pm_cache', rate_limit_s=0.25)
api = client.PolymarketClient(cfg)

markets = api.resolved_markets(limit=3000, min_volume=5_000)
print(f'{len(markets):,} resolved markets with >$5k volume')
print(markets[['question','winning_index','volume']].head(5).to_string(index=False))

**Check the shape before trusting anything.** If the API has drifted, the
adapter says so here rather than three transformations downstream. Fix
`TRADE_FIELDS` / `MARKET_FIELDS` in `src/polymarket/client.py` if needed.

In [ ]:
probe = api.market_trades(markets['market_id'].iloc[0], limit=5)
print('raw fields:', sorted(pd.DataFrame(probe).columns))
client.validate_trade_fields(pd.DataFrame(probe))
print('\nshape OK')

## 3. Discover the wallet population

Start with 200 markets to check the pipeline end to end, then raise it. The
number that matters downstream is `n_wallets_discovered` — the population
**searched**, not the number surviving later filters. Counting after filtering
is the most common way the luck correction gets understated.

In [ ]:
raw_trades, meta = client.discover_population(api, markets, n_markets=200, seed=0)
for k, v in meta.items():
    print(f'{k:24} {v:,}' if isinstance(v, int) else f'{k:24} {v}')

N_SCANNED = meta['n_wallets_discovered']   # carry this to the luck correction

In [ ]:
trades = client.normalise_trades(raw_trades, markets)
print(f'{len(trades):,} resolved trades | {trades.wallet.nunique():,} wallets')
trades.head(3)

## 4. Score every wallet

Edge is `(outcome − price)`, never win rate. A wallet buying only 90c favourites
wins 90% of the time with exactly zero edge.

The luck bar is set on the **t-statistic**, not on cents, because real wallet
populations are wildly heterogeneous — one trader has 20 fills, another 2,000 —
and a cents threshold is not comparable across them.

In [ ]:
scored = wallets.score_wallets(trades, min_trades=20)
ranked = wallets.luck_adjusted_ranking(scored, n_wallets_scanned=N_SCANNED)

print(f'wallets with >=20 resolved trades: {len(ranked):,} of {N_SCANNED:,} discovered')
print(f't-stat needed to clear luck: {ranked.t_needed.iloc[0]:.2f}')
print()
print(ranked.head(15)[['wallet','n_trades','n_eff','edge_per_share','roi',
                       't_stat','clears_luck']].to_string(index=False))

In [ ]:
n_clear = int(ranked.clears_luck.sum())
print(f'>>> wallets clearing the luck bar: {n_clear}')
print()
if n_clear == 0:
    print('No wallet is distinguishable from the luckiest of the population.')
    print('That is a complete answer: there is nothing here safe to copy.')
else:
    print(ranked[ranked.clears_luck][['wallet','n_trades','edge_per_share',
                                      't_stat','roi']].to_string(index=False))
    print('\nNOTE: measured false-positive rate of this gate is ~20% of')
    print('populations, not the nominal 5%. Clearing it is NECESSARY, not')
    print('SUFFICIENT. Sections 5 and 6 are the real tests.')

## 5. Persistence — the decisive test

Rank on one period, measure on the next. If wallets selected for past
performance do not outperform afterwards, past profit carries no information
about future profit and copy trading has nothing to copy.

The split is on **resolution time, not trade time**. A trade placed in period A
on a market resolving in period B has an outcome nobody knew when ranking at the
end of A — splitting on trade time is lookahead.

In [ ]:
pt = wallets.persistence_test(trades, top_frac=0.10)   # split_on='resolved_at'
for k, v in pt.items():
    print(f'{k:26} {v}')

**How to read it.** `gap` is the out-of-sample edge advantage of the top decile
over everyone else. A `gap_t_stat` above 2 with a positive gap is real evidence.
Anything less is not.

Calibration from the synthetic tests: this detects a 15c edge cleanly (t≈6) and
**cannot resolve a 6c edge** at ~1,200 wallets with 60–240 trades each. So a null
result here means *"no edge large enough to matter"*, not *"no edge at all"* —
and since only a large edge is copyable, that is the relevant question anyway.

## 6. Is the edge just favourite-longshot bias?

Longshots are systematically overpriced in prediction markets — documented for
decades. A wallet mechanically fading them looks skilled while harvesting a known
effect.

That distinction decides what you build. If the edge is bias harvesting, **run
the bias directly as a rule** — copying is a strictly worse wrapper that pays
latency and slippage for the same trade.

In [ ]:
top = ranked.head(5).wallet.tolist()
for w in top:
    out = wallets.bias_attribution(trades, w)
    print(f'{w[:12]}...  edge {out.get("overall_edge")}  '
          f'extreme-band stake {out.get("stake_in_extreme_bands")}')
    print(f'   {out.get("verdict")}')

## 7. If anything survived: is it worth copying?

Execution economics. Note the cross-venue penalty is **not** a modelling choice —
US persons may trade Polymarket US but not Global, where these wallets live.

In [ ]:
best = ranked[ranked.clears_luck]
if len(best):
    edge_c = float(best.edge_per_share.iloc[0] * 100)
    for k, v in execution.copy_economics(edge_c).items():
        print(f'{k:30} {v}')
    print()
    print(execution.sensitivity(leader_edge_cents=edge_c).to_string(index=False))
else:
    print('Nothing cleared the luck bar; execution economics are moot.')

## 8. Scale up, and what the answer means

If the pipeline works, raise `n_markets` to 1000+ for a real population. The
crawl is cached and resumable.

- **Nothing clears the bar** → the honest answer, reached for free. Record it and
  stop. Most of the profit on a public leaderboard is the maximum of thousands
  of random walks.
- **Something clears it and persists** → check §6 first. If it is bias
  harvesting, build the rule, not the copier. If it is not, then the geo problem
  becomes worth solving — and that is when to reformulate around Polymarket US.

Either way, record the outcome in the hypothesis registry so the trial count
stays honest.